# Churn Modeling 07-5 - Paper-Based Feature Engineering

`ott_churn_feature_engineering_plan.md`의 논문 기반 방향을 현재 07 계열 실험 흐름에 맞춰 검증한다.

목적은 feature를 많이 추가하는 것이 아니라, 글로벌 SVOD churn 요인인 **가격 대비 가치**, **추천/개인화 반응**, **전환 위험**, **구독 피로감**, **몰아보기/콘텐츠 고갈 proxy**를 leakage-safe transformer로 만들어 `original` feature set과 비교하는 것이다.

## 피쳐 설명

이 노트북의 `paper_fe`는 논문에서 제시된 글로벌 SVOD 해지 요인을 현재 데이터셋 컬럼으로 대체 표현한 파생 피쳐다. 직접 관측되지 않는 개념은 proxy로 만들고, median/quantile 기준은 train split의 `fit()` 단계에서만 계산한다.

| 구분 | 피쳐명 | 설명 |
| --- | --- | --- |
| 요금제 | `plan_tier` | `subscription_type`을 Basic=1, Standard=2, Premium=3으로 변환한 요금제 등급 |
| RFM 변형 | `recency_score` | 마지막 로그인 이후 경과 일수. 값이 클수록 최근성이 낮음 |
| RFM 변형 | `frequency_score` | 주당 시청 세션 수 |
| RFM 변형 | `monetary_score` | 요금제 등급 proxy |
| RFM 변형 | `rfm_churn_risk` | 미접속 기간, 시청 빈도, 요금제 등급을 결합한 해지 위험 점수 |
| 가격 대비 가치 | `watch_time_per_plan` | 요금제 등급 대비 주당 시청 시간 |
| 가격 대비 가치 | `sessions_per_plan` | 요금제 등급 대비 주당 시청 세션 수 |
| 가격 대비 가치 | `completion_per_plan` | 요금제 등급 대비 콘텐츠 완료율 |
| 가격 대비 가치 | `price_burden_proxy` | 시청 시간이 낮은 고요금제 사용자일수록 커지는 가격 부담 proxy |
| 가격 대비 가치 | `high_plan_low_usage` | Premium 사용자 중 시청 시간이 train 중앙값보다 낮은 사용자 flag |
| 가격 대비 가치 | `value_for_money_score` | 시청 시간, 세션 수, 완료율을 요금제 등급으로 나눈 가격 대비 이용 가치 점수 |
| 추천/개인화 | `recommendation_effectiveness` | 추천 클릭률과 완료율을 결합한 추천 소비 전환 proxy |
| 추천/개인화 | `recommendation_mismatch` | 추천 클릭률과 완료율의 차이. 추천 클릭 후 만족 소비로 이어지지 않을 가능성 |
| 추천/개인화 | `personalization_risk` | 추천 클릭률과 앱 평가가 모두 train 중앙값보다 낮은 사용자 flag |
| 추천/개인화 | `is_algorithm_source` | 추천/유입 경로가 Algorithm인지 여부 |
| 추천/개인화 | `recommendation_satisfaction_proxy` | 추천 클릭률, 완료율, 평균 평점을 결합한 추천 만족도 proxy |
| 전환 위험 | `switching_risk_proxy` | 긴 미접속, 낮은 시청량, 낮은 추천 반응을 동시에 보이는 사용자 flag |
| 서비스 필요성 | `service_dependency_score` | 시청 세션, 접속 세션, 시청 시간을 p95 기준으로 정규화해 결합한 서비스 의존도 점수 |
| 서비스 필요성 | `low_service_need` | 서비스 의존도 점수가 train 중앙값보다 낮은 사용자 flag |
| 구독 피로감 | `subscription_fatigue_proxy` | 오래 구독했지만 최근 미접속과 낮은 시청 빈도를 보이는 사용자 flag |
| 구독 피로감 | `usage_per_account_age` | 계정 연령 대비 현재 주당 시청 시간 |
| 몰아보기 | `avg_minutes_per_watch_session` | 시청 세션당 평균 시청 시간 |
| 몰아보기 | `binge_intensity_proxy` | 세션당 평균 시청 시간과 완료율을 결합한 몰아보기 proxy |
| 콘텐츠 고갈 | `content_exhaustion_proxy` | 시청 시간이 높고 완료율도 높은 사용자 flag |
| 논문 범주 | `is_tv_device` | 주 시청 기기가 Smart TV인지 여부 |
| 논문 범주 | `is_retention_genre` | 선호 장르가 Drama 또는 Comedy인지 여부 |
| 종합 위험 | `paper_risk_count` | 논문 기반 위험 flag들의 합계 |

주의: `days_since_last_login`, `app_rating`, `avg_rating_given`, `recommendation_click_rate`, `completion_rate`는 churn 기준 시점 이후 정보가 섞이면 target leakage가 될 수 있다. 현재 실험은 해당 컬럼들이 기준 시점 이전 행동 집계값이라는 전제로 진행한다.

## 1. Imports

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS = -1
MODEL_N_JOBS = -1
N_ITER_FAST = 6

In [2]:
def make_ohe(**kwargs):
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False, **kwargs)

## 2. Load Data and Split

`07-3`, `07-4`와 같은 80/10/10 split을 사용한다. threshold는 valid set에서 고르고 test set에는 그대로 적용한다.

In [3]:
data_path = Path('../data/user_behavior_50000/netflix_user_behavior_churn_50000.csv')
if not data_path.exists():
    data_path = Path('data/user_behavior_50000/netflix_user_behavior_churn_50000.csv')

df = pd.read_csv(data_path)

X = df.drop(columns=['user_id', 'churned'])
y = df['churned']

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=1 / 9,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

print('Train shape:', X_train.shape, 'Valid shape:', X_valid.shape, 'Test shape:', X_test.shape)
print('Train churn ratio:', round(y_train.mean(), 4))
print('Valid churn ratio:', round(y_valid.mean(), 4))
print('Test churn ratio:', round(y_test.mean(), 4))
X_train.head()

Train shape: (40000, 18) Valid shape: (5000, 18) Test shape: (5000, 18)
Train churn ratio: 0.21
Valid churn ratio: 0.21
Test churn ratio: 0.21


,age,gender,region,subscription_type,payment_method,primary_device,account_age_months,favorite_genre,time_of_day,recommendation_source,session_count,avg_watch_time_minutes_per_week,watch_sessions_per_week,completion_rate,avg_rating_given,app_rating,recommendation_click_rate,days_since_last_login
34699,41,Male,North America,Basic,Credit Card,Smart TV,54,Drama,Night,Trending,1,444,8,70,3,4,42,42
30617,34,Male,North America,Basic,Credit Card,Smart TV,18,Romance,Evening,Algorithm,1,170,4,67,4,4,69,0
28850,22,Male,North America,Basic,Paypal,Smart TV,43,Documentary,Afternoon,Ads,1,342,7,76,5,4,18,3
46436,33,Male,North America,Basic,Debit Card,Mobile,39,Documentary,Morning,Ads,1,121,4,60,3,2,0,28
38655,24,Male,South America,Basic,Credit Card,Laptop,1,Horror,Evening,Friend,1,119,1,61,3,4,28,0


### 결과 해석 - Split 확인

- 전체 50,000건을 train 40,000건, valid 5,000건, test 5,000건으로 나눴다.
- 세 구간의 churn 비율이 모두 0.21로 동일하게 유지됐다. stratify split이 정상적으로 적용됐기 때문에, 이후 valid threshold 선택과 test 평가를 비교할 수 있는 기본 조건은 맞다.
- 원본 입력 feature는 `user_id`, `churned`를 제외한 18개다. `paper_fe`는 이 18개를 보존한 상태에서 파생변수를 추가하는 구조다.

## 3. Category Check

논문 기반 범주 피쳐(`Smart TV`, `Drama`, `Comedy`, `Algorithm`)가 실제 데이터 값과 맞는지 먼저 확인한다.

In [4]:
category_cols = [
    'subscription_type', 'primary_device', 'favorite_genre',
    'recommendation_source', 'payment_method', 'region', 'time_of_day',
]

for col in category_cols:
    print(f'\n[{col}]')
    print(X_train[col].value_counts(dropna=False).to_string())


[subscription_type]
subscription_type
Basic       15160
Standard    14444
Premium     10396

[primary_device]
primary_device
Smart TV    19137
Mobile      12019
Laptop       5649
Tablet       3195

[favorite_genre]
favorite_genre
Comedy         8026
Drama          7192
Action         6405
Documentary    5622
Sci-Fi         5202
Horror         3985
Romance        3568

[recommendation_source]
recommendation_source
Algorithm    13650
Trending      9979
Search        6409
Friend        5992
Ads           3970

[payment_method]
payment_method
Credit Card    17207
Paypal         10726
Debit Card      8828
Gift Card       3239

[region]
region
Asia             12800
North America     9563
Europe            8065
South America     4378
Africa            3598
Oceania           1596

[time_of_day]
time_of_day
Evening      16820
Night        10819
Afternoon     7954
Morning       4407


### 결과 해석 - 범주값 확인

- `subscription_type`은 `Basic`, `Standard`, `Premium`만 존재해 `plan_tier` 매핑을 그대로 적용할 수 있다.
- `primary_device`에는 `Smart TV`가 가장 많고, 논문 기반 `is_tv_device` 생성 기준과 실제 값이 일치한다.
- `favorite_genre`에는 `Comedy`, `Drama`가 모두 존재해 `is_retention_genre = Drama 또는 Comedy` 기준을 적용할 수 있다.
- `recommendation_source`에는 `Algorithm`이 존재해 `is_algorithm_source`를 만들 수 있다.
- 즉, md 문서에서 가정한 주요 범주값은 현재 데이터셋과 충돌하지 않는다.

## 4. Paper-Based Feature Engineer

md 문서의 피쳐를 현재 실험에 맞춰 조정한다.

- quantile/median 임계값은 `fit()`에서 train 기준으로 저장한다.
- `recommendation_click_rate`, `completion_rate`는 0~100 스케일이므로 곱셈 피쳐도 0~100 범위로 맞춘다.
- RFM 수동 점수는 raw scale이 한 변수에 끌려가지 않도록 p95 기반 normalization을 사용한다.
- target leakage 가능성이 있는 행동 컬럼은 기준 시점 이전 집계 데이터라는 전제가 필요하다.

In [5]:
class PaperBasedChurnFeatureEngineer(BaseEstimator, TransformerMixin):
    new_features_ = [
        'plan_tier', 'recency_score', 'frequency_score', 'monetary_score', 'rfm_churn_risk',
        'watch_time_per_plan', 'sessions_per_plan', 'completion_per_plan', 'price_burden_proxy',
        'high_plan_low_usage', 'value_for_money_score', 'recommendation_effectiveness',
        'recommendation_mismatch', 'personalization_risk', 'is_algorithm_source',
        'recommendation_satisfaction_proxy', 'switching_risk_proxy', 'service_dependency_score',
        'low_service_need', 'subscription_fatigue_proxy', 'usage_per_account_age',
        'avg_minutes_per_watch_session', 'binge_intensity_proxy', 'content_exhaustion_proxy',
        'is_tv_device', 'is_retention_genre', 'paper_risk_count',
    ]

    def __init__(self, eps=1e-6):
        self.eps = eps
        self.plan_map = {'Basic': 1, 'Standard': 2, 'Premium': 3}

    def _plan_tier(self, X):
        plan = X['subscription_type'].map(self.plan_map).astype(float)
        return plan.fillna(self.plan_tier_median_)

    def _service_dependency_score(self, X):
        watch_norm = np.clip(X['avg_watch_time_minutes_per_week'] / self.watch_p95_, 0, 1)
        weekly_session_norm = np.clip(X['watch_sessions_per_week'] / self.weekly_session_p95_, 0, 1)
        session_count_norm = np.clip(X['session_count'] / self.session_count_p95_, 0, 1)
        return 100 * (0.40 * weekly_session_norm + 0.30 * session_count_norm + 0.30 * watch_norm)

    def fit(self, X, y=None):
        X_fit = X.copy()
        plan_tier = X_fit['subscription_type'].map(self.plan_map).astype(float)
        self.plan_tier_median_ = plan_tier.median()
        if pd.isna(self.plan_tier_median_):
            self.plan_tier_median_ = 2.0

        self.watch_median_ = X_fit['avg_watch_time_minutes_per_week'].median()
        self.watch_high_q_ = X_fit['avg_watch_time_minutes_per_week'].quantile(0.75)
        self.weekly_session_median_ = X_fit['watch_sessions_per_week'].median()
        self.completion_high_q_ = X_fit['completion_rate'].quantile(0.75)
        self.reco_median_ = X_fit['recommendation_click_rate'].median()
        self.app_rating_median_ = X_fit['app_rating'].median()
        self.inactive_median_ = X_fit['days_since_last_login'].median()
        self.account_age_median_ = X_fit['account_age_months'].median()
        self.watch_p95_ = max(X_fit['avg_watch_time_minutes_per_week'].quantile(0.95), self.eps)
        self.weekly_session_p95_ = max(X_fit['watch_sessions_per_week'].quantile(0.95), self.eps)
        self.session_count_p95_ = max(X_fit['session_count'].quantile(0.95), self.eps)
        self.inactive_p95_ = max(X_fit['days_since_last_login'].quantile(0.95), self.eps)
        self.service_dependency_median_ = np.median(self._service_dependency_score(X_fit))
        return self

    def transform(self, X):
        X_fe = X.copy()
        eps = self.eps
        X_fe['plan_tier'] = self._plan_tier(X_fe)

        # Leakage-risk source columns: days_since_last_login, ratings, recommendation_click_rate, completion_rate.
        X_fe['recency_score'] = X_fe['days_since_last_login']
        X_fe['frequency_score'] = X_fe['watch_sessions_per_week']
        X_fe['monetary_score'] = X_fe['plan_tier']
        recency_norm = np.clip(X_fe['days_since_last_login'] / self.inactive_p95_, 0, 1)
        frequency_norm = np.clip(X_fe['watch_sessions_per_week'] / self.weekly_session_p95_, 0, 1)
        monetary_norm = np.clip((X_fe['plan_tier'] - 1) / 2, 0, 1)
        X_fe['rfm_churn_risk'] = 100 * (0.50 * recency_norm - 0.30 * frequency_norm + 0.20 * monetary_norm)

        X_fe['watch_time_per_plan'] = X_fe['avg_watch_time_minutes_per_week'] / (X_fe['plan_tier'] + eps)
        X_fe['sessions_per_plan'] = X_fe['watch_sessions_per_week'] / (X_fe['plan_tier'] + eps)
        X_fe['completion_per_plan'] = X_fe['completion_rate'] / (X_fe['plan_tier'] + eps)
        X_fe['price_burden_proxy'] = X_fe['plan_tier'] / np.log1p(X_fe['avg_watch_time_minutes_per_week'] + eps)
        X_fe['high_plan_low_usage'] = (
            (X_fe['plan_tier'] >= 3) & (X_fe['avg_watch_time_minutes_per_week'] < self.watch_median_)
        ).astype(int)
        X_fe['value_for_money_score'] = (
            X_fe['avg_watch_time_minutes_per_week'] + X_fe['watch_sessions_per_week'] * 10 + X_fe['completion_rate']
        ) / (X_fe['plan_tier'] + eps)

        X_fe['recommendation_effectiveness'] = (
            X_fe['recommendation_click_rate'] / 100 * X_fe['completion_rate'] / 100 * 100
        )
        X_fe['recommendation_mismatch'] = X_fe['recommendation_click_rate'] - X_fe['completion_rate']
        X_fe['personalization_risk'] = (
            (X_fe['recommendation_click_rate'] < self.reco_median_) & (X_fe['app_rating'] < self.app_rating_median_)
        ).astype(int)
        X_fe['is_algorithm_source'] = (X_fe['recommendation_source'].astype(str) == 'Algorithm').astype(int)
        X_fe['recommendation_satisfaction_proxy'] = (
            X_fe['recommendation_click_rate'] + X_fe['completion_rate'] + X_fe['avg_rating_given'] * 20
        ) / 3

        X_fe['switching_risk_proxy'] = (
            (X_fe['days_since_last_login'] > self.inactive_median_)
            & (X_fe['avg_watch_time_minutes_per_week'] < self.watch_median_)
            & (X_fe['recommendation_click_rate'] < self.reco_median_)
        ).astype(int)
        X_fe['service_dependency_score'] = self._service_dependency_score(X_fe)
        X_fe['low_service_need'] = (X_fe['service_dependency_score'] < self.service_dependency_median_).astype(int)
        X_fe['subscription_fatigue_proxy'] = (
            (X_fe['account_age_months'] > self.account_age_median_)
            & (X_fe['days_since_last_login'] > self.inactive_median_)
            & (X_fe['watch_sessions_per_week'] < self.weekly_session_median_)
        ).astype(int)
        X_fe['usage_per_account_age'] = X_fe['avg_watch_time_minutes_per_week'] / (X_fe['account_age_months'] + 1)
        X_fe['avg_minutes_per_watch_session'] = X_fe['avg_watch_time_minutes_per_week'] / (X_fe['watch_sessions_per_week'] + eps)
        X_fe['binge_intensity_proxy'] = X_fe['avg_minutes_per_watch_session'] * (X_fe['completion_rate'] / 100)
        X_fe['content_exhaustion_proxy'] = (
            (X_fe['avg_watch_time_minutes_per_week'] > self.watch_high_q_)
            & (X_fe['completion_rate'] > self.completion_high_q_)
        ).astype(int)
        X_fe['is_tv_device'] = (X_fe['primary_device'].astype(str) == 'Smart TV').astype(int)
        X_fe['is_retention_genre'] = X_fe['favorite_genre'].isin(['Drama', 'Comedy']).astype(int)

        risk_cols = [
            'high_plan_low_usage', 'personalization_risk', 'switching_risk_proxy', 'low_service_need',
            'subscription_fatigue_proxy', 'content_exhaustion_proxy', 'is_tv_device',
        ]
        X_fe['paper_risk_count'] = X_fe[risk_cols].sum(axis=1)
        return X_fe

## 5. Feature Sanity Check

In [6]:
paper_debugger = PaperBasedChurnFeatureEngineer()
X_train_paper = paper_debugger.fit_transform(X_train)

paper_added_cols = sorted(set(X_train_paper.columns) - set(X_train.columns))
print('Original feature count:', X_train.shape[1])
print('Paper FE feature count:', X_train_paper.shape[1])
print('Added feature count:', len(paper_added_cols))
print(paper_added_cols)

missing = X_train_paper[paper_debugger.new_features_].isna().sum()
print('\nMissing values')
print(missing[missing > 0])

numeric_paper = X_train_paper[paper_debugger.new_features_].select_dtypes(include='number')
inf_count = np.isinf(numeric_paper).sum()
print('\nInfinite values')
print(inf_count[inf_count > 0])

X_train_paper[paper_debugger.new_features_].head()

Original feature count: 18
Paper FE feature count: 45
Added feature count: 27
['avg_minutes_per_watch_session', 'binge_intensity_proxy', 'completion_per_plan', 'content_exhaustion_proxy', 'frequency_score', 'high_plan_low_usage', 'is_algorithm_source', 'is_retention_genre', 'is_tv_device', 'low_service_need', 'monetary_score', 'paper_risk_count', 'personalization_risk', 'plan_tier', 'price_burden_proxy', 'recency_score', 'recommendation_effectiveness', 'recommendation_mismatch', 'recommendation_satisfaction_proxy', 'rfm_churn_risk', 'service_dependency_score', 'sessions_per_plan', 'subscription_fatigue_proxy', 'switching_risk_proxy', 'usage_per_account_age', 'value_for_money_score', 'watch_time_per_plan']

Missing values
Series([], dtype: int64)

Infinite values
Series([], dtype: int64)


,plan_tier,recency_score,frequency_score,monetary_score,rfm_churn_risk,watch_time_per_plan,sessions_per_plan,completion_per_plan,price_burden_proxy,high_plan_low_usage,...,service_dependency_score,low_service_need,subscription_fatigue_proxy,usage_per_account_age,avg_minutes_per_watch_session,binge_intensity_proxy,content_exhaustion_proxy,is_tv_device,is_retention_genre,paper_risk_count
34699,1.0,42,8,1.0,28.205128,443.999556,7.999992,69.999930,0.163986,0,...,54.131914,0,0,8.072727,55.499993,38.849995,0,1,1,1
30617,1.0,0,4,1.0,-9.230769,169.999830,3.999996,66.999933,0.194490,0,...,28.237444,1,0,8.947368,42.499989,28.474993,0,1,0,2
28850,1.0,3,7,1.0,-12.820513,341.999658,6.999993,75.999924,0.171299,0,...,45.997139,0,0,7.772727,48.857136,37.131423,0,1,0,1
46436,1.0,28,4,1.0,21.880342,120.999879,3.999996,59.999940,0.208159,0,...,25.807692,1,0,3.025000,30.249992,18.149995,0,0,0,3
38655,1.0,0,1,1.0,-2.307692,118.999881,0.999999,60.999939,0.208878,0,...,16.477750,1,0,59.500000,118.999881,72.589927,0,0,0,1


### 결과 해석 - Paper FE 생성 점검

- 원본 18개 feature에서 `paper_fe` 적용 후 45개 feature가 됐다. 새로 추가된 파생변수는 27개다.
- 추가 피쳐는 가격 대비 가치, 추천 반응, 서비스 의존도, 전환 위험, 구독 피로감, 몰아보기 proxy, 논문 기반 범주 flag를 모두 포함한다.
- 결측치와 무한대 값이 모두 0건으로 확인됐다. 따라서 모델 입력 단계에서 `paper_fe` 때문에 별도 결측/inf 처리를 추가할 필요는 없다.
- `recommendation_effectiveness`, `binge_intensity_proxy`처럼 곱셈 기반 피쳐도 정상 범위로 생성됐다.
- 이 단계에서 확인한 것은 생성 안정성이지 성능 개선 여부는 아니다. 성능 채택 여부는 뒤의 valid/test 비교를 기준으로 판단한다.

## 6. Pipeline and Metrics

In [7]:
def build_preprocessor(X_schema):
    numeric_features = X_schema.select_dtypes(include='number').columns.tolist()
    categorical_features = X_schema.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', make_ohe(), categorical_features),
        ],
        remainder='drop',
    )


def make_feature_engineer(feature_set):
    if feature_set == 'paper_fe':
        return PaperBasedChurnFeatureEngineer()
    return None


def build_pipeline(X_train_schema, estimator, feature_set='original'):
    fe = make_feature_engineer(feature_set)
    steps = []
    if fe is not None:
        X_schema = fe.fit_transform(X_train_schema)
        steps.append(('feature_engineering', fe))
    else:
        X_schema = X_train_schema.copy()
    steps.extend([('preprocess', build_preprocessor(X_schema)), ('model', estimator)])
    return Pipeline(steps)


def get_scores(model, X_eval):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_eval)[:, 1]
    raw_scores = model.decision_function(X_eval)
    return (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-9)


def tune_threshold_for_f1(y_true, y_score, thresholds=np.arange(0.05, 0.951, 0.005)):
    rows = []
    for threshold in thresholds:
        y_pred = (np.asarray(y_score) >= threshold).astype(int)
        rows.append({
            'threshold': threshold,
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
        })
    threshold_df = pd.DataFrame(rows)
    return threshold_df.loc[threshold_df['f1'].idxmax()], threshold_df


def evaluate_auc_f1(y_true, y_score, threshold=0.5):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }

## 7. Model Spaces

`07-5`는 feature engineering 비교가 목적이므로 탐색 폭은 가볍게 둔다. 필요하면 `N_ITER_FAST`를 10~15로 늘려 재실험한다.

In [8]:
def available_f1_model_spaces(y_train):
    spaces = {}
    spaces['LogisticRegression'] = {
        'estimator': LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        'params': {
            'model__C': np.logspace(-2, 2, 10),
            'model__class_weight': [None, 'balanced'],
            'model__solver': ['lbfgs'],
        },
    }

    try:
        from lightgbm import LGBMClassifier
        spaces['LightGBM'] = {
            'estimator': LGBMClassifier(random_state=RANDOM_STATE, n_jobs=MODEL_N_JOBS, verbose=-1),
            'params': {
                'model__n_estimators': [150, 250, 400],
                'model__learning_rate': [0.03, 0.05, 0.08],
                'model__num_leaves': [15, 31, 63],
                'model__max_depth': [-1, 4, 6],
                'model__min_child_samples': [20, 50, 100],
                'model__subsample': [0.8, 1.0],
                'model__colsample_bytree': [0.8, 1.0],
                'model__class_weight': [None, 'balanced'],
            },
        }
    except Exception as exc:
        print('[skip] LightGBM unavailable:', exc)

    try:
        from xgboost import XGBClassifier
        neg, pos = np.bincount(y_train)
        spaces['XGBoost'] = {
            'estimator': XGBClassifier(
                objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE,
                n_jobs=MODEL_N_JOBS, tree_method='hist',
            ),
            'params': {
                'model__n_estimators': [150, 250, 400],
                'model__learning_rate': [0.03, 0.05, 0.08],
                'model__max_depth': [3, 4, 5],
                'model__min_child_weight': [1, 3, 5],
                'model__subsample': [0.8, 1.0],
                'model__colsample_bytree': [0.8, 1.0],
                'model__scale_pos_weight': [1, neg / pos],
            },
        }
    except Exception as exc:
        print('[skip] XGBoost unavailable:', exc)

    try:
        from catboost import CatBoostClassifier
        spaces['CatBoost'] = {
            'estimator': CatBoostClassifier(
                loss_function='Logloss', eval_metric='F1', random_seed=RANDOM_STATE,
                verbose=False, allow_writing_files=False, thread_count=MODEL_N_JOBS,
            ),
            'params': {
                'model__iterations': [150, 250, 400],
                'model__learning_rate': [0.03, 0.05, 0.08],
                'model__depth': [4, 6, 8],
                'model__l2_leaf_reg': [1, 3, 5, 7],
                'model__auto_class_weights': [None, 'Balanced'],
            },
        }
    except Exception as exc:
        print('[skip] CatBoost unavailable:', exc)
    return spaces

model_spaces = available_f1_model_spaces(y_train)
print('Model candidates:', list(model_spaces.keys()))

Model candidates: ['LogisticRegression', 'LightGBM', 'XGBoost', 'CatBoost']


### 결과 해석 - 모델 후보

- 비교 후보는 `LogisticRegression`, `LightGBM`, `XGBoost`, `CatBoost` 네 가지가 정상적으로 구성됐다.
- 이번 노트북의 비교 축은 모델 자체보다 feature set이다. 따라서 각 모델에서 `original`과 `paper_fe`를 같은 search 조건으로 비교하는 방식이 적절하다.
- `N_ITER_FAST = 6`이므로 이 결과는 가벼운 비교 실험이다. 큰 방향성은 볼 수 있지만, 최종 튜닝 결과로 과해석하지 않는다.

## 8. F1 Search: Original vs Paper FE

`original`과 `paper_fe`만 비교한다. valid set에서 F1 threshold를 선택하고, test set에서는 valid threshold를 그대로 적용한다.

In [9]:
feature_sets = ['original', 'paper_fe']
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {'f1': 'f1', 'roc_auc': 'roc_auc', 'average_precision': 'average_precision'}

search_rows = []
fitted_candidates = {}
valid_scores = {}
valid_thresholds = {}

for feature_set in feature_sets:
    for model_name, spec in model_spaces.items():
        run_name = f'{model_name}__{feature_set}'
        pipe = build_pipeline(X_train, spec['estimator'], feature_set=feature_set)
        n_iter = min(N_ITER_FAST, int(np.prod([len(v) for v in spec['params'].values()])))
        print(f'[search] {run_name} | n_iter={n_iter}')
        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=spec['params'],
            n_iter=n_iter,
            scoring=scoring,
            refit='f1',
            cv=cv,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbose=0,
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        y_valid_score = get_scores(best_model, X_valid)
        valid_at_05 = evaluate_auc_f1(y_valid, y_valid_score, threshold=0.5)
        best_threshold_row, _ = tune_threshold_for_f1(y_valid, y_valid_score)
        threshold = float(best_threshold_row['threshold'])
        valid_tuned = evaluate_auc_f1(y_valid, y_valid_score, threshold=threshold)

        fitted_candidates[run_name] = best_model
        valid_scores[run_name] = y_valid_score
        valid_thresholds[run_name] = threshold
        search_rows.append({
            'run_name': run_name,
            'model': model_name,
            'feature_set': feature_set,
            'cv_best_f1': search.best_score_,
            'cv_mean_roc_auc_at_best': search.cv_results_['mean_test_roc_auc'][search.best_index_],
            'cv_mean_pr_auc_at_best': search.cv_results_['mean_test_average_precision'][search.best_index_],
            'valid_roc_auc': valid_at_05['roc_auc'],
            'valid_pr_auc': valid_at_05['pr_auc'],
            'valid_f1_at_05': valid_at_05['f1'],
            'valid_best_threshold': threshold,
            'valid_best_f1': valid_tuned['f1'],
            'valid_precision_at_best_threshold': valid_tuned['precision'],
            'valid_recall_at_best_threshold': valid_tuned['recall'],
            'best_params': search.best_params_,
        })
        print(
            f"[done] {run_name} | CV F1={search.best_score_:.4f}, "
            f"Valid best F1={valid_tuned['f1']:.4f}, threshold={threshold:.3f}, "
            f"Valid ROC AUC={valid_at_05['roc_auc']:.4f}, PR AUC={valid_at_05['pr_auc']:.4f}"
        )

search_result = pd.DataFrame(search_rows).sort_values(
    ['valid_best_f1', 'valid_pr_auc', 'valid_roc_auc'], ascending=False
).reset_index(drop=True)
search_result[[
    'run_name', 'model', 'feature_set', 'cv_best_f1', 'valid_best_f1',
    'valid_best_threshold', 'valid_roc_auc', 'valid_pr_auc',
    'valid_precision_at_best_threshold', 'valid_recall_at_best_threshold',
]]

[search] LogisticRegression__original | n_iter=6
[done] LogisticRegression__original | CV F1=0.6763, Valid best F1=0.7273, threshold=0.710, Valid ROC AUC=0.9219, PR AUC=0.7950
[search] LightGBM__original | n_iter=6
[done] LightGBM__original | CV F1=0.6765, Valid best F1=0.7168, threshold=0.675, Valid ROC AUC=0.9187, PR AUC=0.7925
[search] XGBoost__original | n_iter=6
[done] XGBoost__original | CV F1=0.6762, Valid best F1=0.7169, threshold=0.695, Valid ROC AUC=0.9192, PR AUC=0.7923
[search] CatBoost__original | n_iter=6
[done] CatBoost__original | CV F1=0.6824, Valid best F1=0.7219, threshold=0.670, Valid ROC AUC=0.9193, PR AUC=0.7922
[search] LogisticRegression__paper_fe | n_iter=6
[done] LogisticRegression__paper_fe | CV F1=0.6761, Valid best F1=0.7232, threshold=0.720, Valid ROC AUC=0.9218, PR AUC=0.7948
[search] LightGBM__paper_fe | n_iter=6
[done] LightGBM__paper_fe | CV F1=0.6768, Valid best F1=0.7132, threshold=0.625, Valid ROC AUC=0.9153, PR AUC=0.7842
[search] XGBoost__paper_fe

,run_name,model,feature_set,cv_best_f1,valid_best_f1,valid_best_threshold,valid_roc_auc,valid_pr_auc,valid_precision_at_best_threshold,valid_recall_at_best_threshold
0,LogisticRegression__original,LogisticRegression,original,0.676336,0.727273,0.710,0.921869,0.795017,0.734694,0.720000
1,LogisticRegression__paper_fe,LogisticRegression,paper_fe,0.676077,0.723175,0.720,0.921800,0.794824,0.744702,0.702857
2,CatBoost__original,CatBoost,original,0.682438,0.721910,0.670,0.919265,0.792197,0.709945,0.734286
3,CatBoost__paper_fe,CatBoost,paper_fe,0.679512,0.720624,0.700,0.919852,0.791441,0.738262,0.703810
4,XGBoost__paper_fe,XGBoost,paper_fe,0.676708,0.718535,0.660,0.919349,0.791471,0.691630,0.747619
5,XGBoost__original,XGBoost,original,0.676181,0.716873,0.695,0.919233,0.792336,0.717557,0.716190
6,LightGBM__original,LightGBM,original,0.676453,0.716806,0.675,0.918675,0.792504,0.699275,0.735238
7,LightGBM__paper_fe,LightGBM,paper_fe,0.676838,0.713192,0.625,0.915300,0.784235,0.676345,0.754286


### 결과 해석 - Valid 성능 비교

- valid 기준 1위는 `LogisticRegression__original`이다. valid best F1은 0.7273, ROC AUC는 0.9219, PR AUC는 0.7950이다.
- `LogisticRegression__paper_fe`는 valid best F1 0.7232로 2위다. original 대비 F1은 약 0.0041 낮고, ROC AUC와 PR AUC도 아주 소폭 낮다.
- tree/boosting 계열에서도 `paper_fe`가 일관된 개선을 만들지는 못했다. `XGBoost__paper_fe`는 같은 XGBoost original보다 valid F1은 약간 높지만, 전체 1위에는 못 미친다.
- `LightGBM__paper_fe`는 valid ROC AUC 0.9153, PR AUC 0.7842로 paper feature 추가 후 ranking 품질이 눈에 띄게 낮아졌다.
- valid 결과만 보면 `paper_fe`는 설명 가능한 신호를 추가했지만, 예측 성능에서는 original feature set을 확실히 이기지 못했다.
- threshold는 모델별로 0.625~0.720 사이에서 선택됐다. 기본 0.5보다 높은 threshold가 반복적으로 선택되므로, 이 데이터에서는 F1 최적화를 위해 보수적인 양성 판정 기준이 필요하다.

## 9. Test Evaluation

valid 기준 상위 후보만 test set에서 확인한다. test set으로 threshold를 다시 고르지 않는다.

In [10]:
top_run_names = search_result['run_name'].tolist()

test_rows = []
test_scores = {}
for run_name in top_run_names:
    model = fitted_candidates[run_name]
    threshold = valid_thresholds[run_name]
    y_test_score = get_scores(model, X_test)
    test_at_05 = evaluate_auc_f1(y_test, y_test_score, threshold=0.5)
    test_tuned = evaluate_auc_f1(y_test, y_test_score, threshold=threshold)
    test_scores[run_name] = y_test_score
    test_rows.append({
        'run_name': run_name,
        'model': run_name.split('__')[0],
        'feature_set': run_name.split('__')[1],
        'valid_selected_threshold': threshold,
        'test_roc_auc': test_at_05['roc_auc'],
        'test_pr_auc': test_at_05['pr_auc'],
        'test_f1_at_05': test_at_05['f1'],
        'test_f1_at_valid_threshold': test_tuned['f1'],
        'test_precision_at_valid_threshold': test_tuned['precision'],
        'test_recall_at_valid_threshold': test_tuned['recall'],
        'test_confusion_matrix_at_valid_threshold': test_tuned['confusion_matrix'],
    })

test_result = pd.DataFrame(test_rows).sort_values(
    ['test_f1_at_valid_threshold', 'test_pr_auc', 'test_roc_auc'], ascending=False
).reset_index(drop=True)
test_result

,run_name,model,feature_set,valid_selected_threshold,test_roc_auc,test_pr_auc,test_f1_at_05,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold,test_confusion_matrix_at_valid_threshold
0,LogisticRegression__original,LogisticRegression,original,0.710,0.903762,0.754898,0.668705,0.687292,0.686965,0.687619,"[[3621, 329], [328, 722]]"
1,CatBoost__original,CatBoost,original,0.670,0.902342,0.752194,0.675069,0.686691,0.666966,0.707619,"[[3579, 371], [307, 743]]"
2,LogisticRegression__paper_fe,LogisticRegression,paper_fe,0.720,0.903607,0.754457,0.671048,0.685770,0.699703,0.672381,"[[3647, 303], [344, 706]]"
3,XGBoost__paper_fe,XGBoost,paper_fe,0.660,0.901327,0.750591,0.672414,0.684545,0.654783,0.717143,"[[3553, 397], [297, 753]]"
4,LightGBM__original,LightGBM,original,0.675,0.901259,0.748525,0.674708,0.684527,0.664574,0.705714,"[[3576, 374], [309, 741]]"
5,CatBoost__paper_fe,CatBoost,paper_fe,0.700,0.902279,0.750799,0.670871,0.679808,0.686408,0.673333,"[[3627, 323], [343, 707]]"
6,LightGBM__paper_fe,LightGBM,paper_fe,0.625,0.897628,0.742212,0.667468,0.679768,0.640034,0.724762,"[[3522, 428], [289, 761]]"
7,XGBoost__original,XGBoost,original,0.695,0.902306,0.753041,0.675000,0.679245,0.672897,0.685714,"[[3600, 350], [330, 720]]"


### 결과 해석 - Test 성능 비교

- test 기준 최고 F1 후보도 `LogisticRegression__original`이다. valid에서 선택한 threshold 0.710을 적용했을 때 test F1은 0.6873이다.
- `LogisticRegression__paper_fe`는 test F1 0.6858로 `paper_fe` 중 가장 좋았다. original 대비 차이는 약 -0.0015로 매우 작지만, 개선은 아니다.
- ROC AUC와 PR AUC도 original이 근소하게 높다. `LogisticRegression__original`은 ROC AUC 0.9038, PR AUC 0.7549이고, `LogisticRegression__paper_fe`는 ROC AUC 0.9036, PR AUC 0.7545다.
- `paper_fe`는 precision을 올리는 대신 recall을 낮추는 방향으로 작동했다. LogisticRegression 기준 precision은 0.6870에서 0.6997로 올랐지만, recall은 0.6876에서 0.6724로 낮아졌다.
- churn 예측에서 더 많은 이탈자를 잡는 것이 중요하다면 이 recall 하락은 불리하다. 반대로 캠페인 비용이 매우 제한적이고 오탐을 줄이는 것이 더 중요하다면 `paper_fe`의 높은 precision은 검토 여지가 있다.
- 전체적으로 test 결과는 `paper_fe`가 성능을 끌어올린 실험이라기보다, 원본 feature set과 거의 비슷한 성능을 내면서 해석 가능한 proxy를 제공한 실험으로 보는 것이 맞다.

## 10. Feature Set Gain Summary

`paper_fe`가 `original` 대비 실제로 이득인지 feature set별 최고 test 성능으로 확인한다.

In [11]:
feature_gain = (
    test_result
    .sort_values(['test_f1_at_valid_threshold', 'test_pr_auc', 'test_roc_auc'], ascending=False)
    .groupby('feature_set', as_index=False)
    .first()
    .sort_values('test_f1_at_valid_threshold', ascending=False)
)
feature_gain[[
    'feature_set', 'run_name', 'test_f1_at_valid_threshold',
    'test_precision_at_valid_threshold', 'test_recall_at_valid_threshold',
    'test_roc_auc', 'test_pr_auc',
]]

,feature_set,run_name,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold,test_roc_auc,test_pr_auc
0,original,LogisticRegression__original,0.687292,0.686965,0.687619,0.903762,0.754898
1,paper_fe,LogisticRegression__paper_fe,0.685770,0.699703,0.672381,0.903607,0.754457


### 결과 해석 - Feature Set별 최고 성능

- feature set 단위 최고 후보를 비교하면 `original`의 최고 F1은 0.6873, `paper_fe`의 최고 F1은 0.6858이다.
- 두 feature set의 차이는 F1 약 0.0015, ROC AUC 약 0.0002, PR AUC 약 0.0004 수준이다. 수치상 original이 앞서지만, 차이는 매우 작다.
- 성능만 기준으로 최종 모델을 고르면 `original`을 유지하는 것이 더 타당하다.
- 다만 `paper_fe`가 크게 망가지지 않았다는 점도 중요하다. 논문 기반 proxy를 추가해도 모델 성능이 크게 훼손되지는 않았으므로, 발표나 해석용 보조 분석에는 사용할 수 있다.
- 최종 운영 모델 입력으로는 `original`, 설명/세그먼트 해석용으로는 `paper_fe` diagnostics를 병행하는 구성이 가장 안전하다.

## 11. Paper Feature Diagnostics

논문 기반 피쳐가 churn class별로 어떤 방향성을 보이는지 중앙값/평균으로 확인한다. 이 표는 성능이 아니라 해석 검증용이다.

In [12]:
X_train_paper_with_y = X_train_paper.copy()
X_train_paper_with_y['churned'] = y_train.values

check_cols = [
    'plan_tier', 'rfm_churn_risk', 'watch_time_per_plan', 'price_burden_proxy',
    'value_for_money_score', 'recommendation_effectiveness', 'recommendation_mismatch',
    'personalization_risk', 'switching_risk_proxy', 'service_dependency_score',
    'low_service_need', 'subscription_fatigue_proxy', 'binge_intensity_proxy',
    'content_exhaustion_proxy', 'is_tv_device', 'is_retention_genre', 'paper_risk_count',
]
median_by_churn = X_train_paper_with_y.groupby('churned')[check_cols].median().T
mean_by_churn = X_train_paper_with_y.groupby('churned')[check_cols].mean().T

diagnostics = median_by_churn.add_prefix('median_churn_').join(mean_by_churn.add_prefix('mean_churn_'))
diagnostics['median_diff_1_minus_0'] = diagnostics['median_churn_1'] - diagnostics['median_churn_0']
diagnostics['mean_diff_1_minus_0'] = diagnostics['mean_churn_1'] - diagnostics['mean_churn_0']
diagnostics.sort_values('mean_diff_1_minus_0', ascending=False)

churned,median_churn_0,median_churn_1,mean_churn_0,mean_churn_1,median_diff_1_minus_0,mean_diff_1_minus_0
rfm_churn_risk,2.478632,27.350427,5.105031,24.374725,24.871795,19.269694
paper_risk_count,1.000000,2.000000,1.449715,2.117381,1.000000,0.667666
switching_risk_proxy,0.000000,0.000000,0.087816,0.465357,0.000000,0.377541
low_service_need,0.000000,1.000000,0.421646,0.793095,1.000000,0.371450
personalization_risk,0.000000,0.000000,0.089114,0.298810,0.000000,0.209696
subscription_fatigue_proxy,0.000000,0.000000,0.072373,0.195119,0.000000,0.122746
is_retention_genre,0.000000,0.000000,0.387532,0.353810,0.000000,-0.033722
price_burden_proxy,0.356540,0.214871,0.357546,0.288997,-0.141669,-0.068550
content_exhaustion_proxy,0.000000,0.000000,0.146677,0.014881,0.000000,-0.131796
is_tv_device,1.000000,0.000000,0.528956,0.288333,-1.000000,-0.240622


### 결과 해석 - Paper Feature 방향성 진단

- churn 사용자는 non-churn 사용자보다 `rfm_churn_risk`가 크게 높다. median 기준 2.48에서 27.35로 상승했고, mean 차이도 약 +19.27이다. 최근 미접속과 낮은 이용 빈도를 결합한 RFM형 위험 점수는 churn 방향을 잘 잡는다.
- `paper_risk_count`도 churn 사용자에서 높다. median은 1개에서 2개로 증가했고, 평균도 1.45에서 2.12로 증가했다. 여러 위험 신호가 동시에 켜지는 사용자가 실제 churn 쪽에 더 많다.
- `low_service_need`는 churn 사용자의 평균이 0.793으로 non-churn의 0.422보다 훨씬 높다. 서비스 의존도가 낮은 사용자를 잡는 proxy가 가장 해석력 있는 신호 중 하나다.
- `switching_risk_proxy`, `personalization_risk`, `subscription_fatigue_proxy`도 churn 사용자 평균이 더 높다. 각각 전환 위험, 개인화 불만족, 구독 피로감 proxy가 churn 방향과 일치한다.
- 반대로 `service_dependency_score`, `recommendation_effectiveness`, `binge_intensity_proxy`, `value_for_money_score`, `watch_time_per_plan`은 churn 사용자에서 낮다. 즉 유지 고객은 더 자주 보고, 추천이 소비로 이어지고, 가격 대비 이용 가치도 높다.
- 논문 방향과 현재 데이터가 완전히 일치하지 않는 부분도 있다. `plan_tier`는 churn 사용자에서 더 낮고, `is_tv_device`도 churn 사용자에서 더 낮다. 이 데이터에서는 Premium/TV 사용자가 더 위험하다는 논문 신호보다, Basic 또는 낮은 이용 강도 쪽 신호가 더 강하게 나타난다.
- `price_burden_proxy`도 churn 사용자에서 낮다. 이 proxy가 plan tier의 영향을 크게 받기 때문에, 현재 데이터처럼 churn 쪽에 Basic 사용자가 많으면 가격 부담 신호로 잘 작동하지 않는다.
- 따라서 `paper_fe` 전체를 성능 개선용으로 채택하기보다는, 방향성이 맞는 신호와 맞지 않는 신호를 구분해야 한다. 이 데이터에서 강한 신호는 RFM 위험, 서비스 필요성 저하, 전환 위험, 추천/완료 반응 저하다.

### 결과 해석 - 최종 결론

- 이번 `07-5`의 결론은 “논문 기반 feature engineering이 예측 성능을 확실히 개선했다”가 아니다.
- 성능 기준으로는 `LogisticRegression__original`이 가장 안정적이다. test F1, ROC AUC, PR AUC 모두 `paper_fe` 최고 후보보다 근소하게 높다.
- `paper_fe`는 성능 개선보다는 해석력 강화에 가깝다. RFM 위험, 서비스 의존도, 추천 효과, 전환 위험, 구독 피로감 같은 개념을 수치화해 churn 고객의 행동 패턴을 설명하는 데 유용하다.
- 최종 모델 후보를 성능 중심으로 고르면 `original`을 유지한다.
- 발표/보고서에서는 `paper_fe` diagnostics를 활용해 “논문에서 제시한 SVOD 해지 요인을 현재 데이터에서 proxy로 구현했고, 그중 RFM 위험과 서비스 필요성 저하가 churn 방향과 가장 잘 맞았다”라고 정리하는 것이 가장 정확하다.
- 추가 실험을 한다면 모든 `paper_fe`를 한 번에 넣기보다, 방향성이 맞았던 피쳐만 선별한 축소 feature set을 따로 검증하는 것이 낫다. 후보는 `rfm_churn_risk`, `paper_risk_count`, `low_service_need`, `switching_risk_proxy`, `personalization_risk`, `subscription_fatigue_proxy`, `service_dependency_score`, `recommendation_effectiveness`, `value_for_money_score`다.

## 12. Conclusion Template

판단 기준:

1. `paper_fe`가 `original`보다 `test_f1_at_valid_threshold`가 높으면 F1 목적에서 채택 후보다.
2. F1은 올랐지만 ROC AUC/PR AUC가 크게 하락하면 ranking 성능을 희생한 trade-off다.
3. 차이가 0.001 내외면 실질 차이는 작으므로 더 단순한 feature set을 우선한다.
4. 논문 피쳐는 설명력 목적에는 유용하지만, 최종 모델 입력은 test 성능으로 결정한다.